## WTUI API를 사용한 데이터 다운로드 
---

**1. import 및 환경설정**

In [9]:
# 필요한 패키지 import 
import pandas as pd
import numpy as np
import os
import time
import json
import warnings
import requests
from io import BytesIO
from tqdm import tqdm
import openpyxl

# 저장 폴더 생성 
os.makedirs('data/raw/wui', exist_ok=True)

# SSL 검증 끄기
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# WDI url
WUI_URL = 'https://worlduncertaintyindex.com/wp-content/uploads/2026/01/WUI_Data.xlsx'
filepath = 'data/raw/wui/wui.xlsx'


**2. 데이터 다운로드**

In [10]:
# 파일 다운로드 
print('WUI 데이터 다운로드 중...')
try : 
    response = requests.get(WUI_URL, verify=False, timeout=60)
    response.raise_for_status()
    with open(filepath, 'wb') as f :
        f.write(response.content)
    print('저장 완료')
except Exception as e : 
    print(f'저장 실패 : {e}')

WUI 데이터 다운로드 중...
저장 완료


**3. 데이터 정리**

In [12]:
# 시트 확인 
check = openpyxl.load_workbook(filepath, read_only=True)
print(f'시트 목록 : {check.sheetnames}')
check.close()

시트 목록 : ['T0', 'F1', 'F2', 'T1', 'T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'T8', 'T9']


In [13]:
# 시트 내용 확인
for sheet in ['T0', 'F1', 'F2', 'T1', 'T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'T8', 'T9'] : 
    df = pd.read_excel('data/raw/wui/wui.xlsx', sheet_name=sheet, nrows = 5)
    print(f'\n{"-"*60}')
    print(f'시트 : {sheet}')
    print(f'컬럼 : {list(df.columns[:5])}....')
    print(df.head(3))


------------------------------------------------------------
시트 : T0
컬럼 : ['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3']....
  Unnamed: 0                                         Unnamed: 1  Unnamed: 2  \
0        Tab                                        Description         NaN   
1         F1  This tab contains the figure for the World Unc...         NaN   
2         F2  This tab contains the figure for the World Tra...         NaN   

           Unnamed: 3  
0              Update  
1  2025Q4 data added.  
2  2025Q4 data added.  

------------------------------------------------------------
시트 : F1
컬럼 : ['Unnamed: 0', 'Unnamed: 1', 'World Uncertainty Index', 'Unnamed: 3', 'Unnamed: 4']....
  Unnamed: 0 Unnamed: 1 World Uncertainty Index  Unnamed: 3  Unnamed: 4  \
0        NaN        NaN    GDP weighted average         NaN         NaN   
1       year      year2                     WUI         NaN         NaN   
2     1990q1       1990                12496.47         NaN    

In [89]:
# 데이터 분류 
WUI_FILE = 'data/raw/wui/wui.xlsx'

df_wui_raw = pd.read_excel(WUI_FILE, sheet_name='T2')
df_wtui_raw = pd.read_excel(WUI_FILE, sheet_name='T8')
df_meta = pd.read_excel(WUI_FILE, sheet_name='T7')

In [94]:
# wui부터 전처리 
df_wui = df_wui_raw.melt(id_vars='year', var_name= 'iso3', value_name='wui')
df_wui['year_num'] = df_wui['year'].str.extract(r'(\d{4})').astype(int)
df_wui['quarter'] = df_wui['year'].str.extract(r'q(\d)').astype(int)


In [95]:
# wtui 전처리 
global_cols = [c for c in df_wtui_raw.columns if 'World' in str(c) or 'Unnamed' in str(c)]
df_wtui = df_wtui_raw.drop(columns=[c for c in global_cols])
df_wtui = df_wtui.melt(id_vars='year', var_name='iso3', value_name='wtui')

df_wtui['year_num'] = df_wtui['year'].str.extract(r'(\d{4})').astype(int)
df_wtui['quarter'] = df_wtui['year'].str.extract(r'q(\d)').astype(int)

In [96]:
# 국가 메타데이터 확인
print(f'메타데이터 : 국가 {len(df_meta)}개국')

메타데이터 : 국가 143개국


In [97]:
# 연도별 평균 내기 
df_wui_annual = (df_wui.groupby(['iso3', 'year_num'])['wui'].mean().reset_index())
df_wtui_annual = (df_wtui.groupby(['iso3', 'year_num'])['wtui'].mean().reset_index())

In [98]:
# WUI-WTUI 병합
df_uncertainty = df_wui_annual.merge(df_wtui_annual, on=['iso3', 'year_num'], how='outer')


In [99]:
# 국가 데이터 병합
df_uncertainty = df_uncertainty.merge(df_meta[['iso_code', 'country_name', 'region']], left_on='iso3', right_on='iso_code', how='left')


In [102]:
# 한국 데이터 가져오기 
kr = df_uncertainty[df_uncertainty['iso3'] == 'KOR']

print(f"   WUI 기간: {kr['year_num'].min()} ~ {kr['year_num'].max()}")
print(f"   WTUI 기간: {kr.dropna(subset=['wtui'])['year_num'].min()} ~ {kr.dropna(subset=['wtui'])['year_num'].max()}")
df_uncertainty = df_uncertainty.rename(columns = {'year_num' : 'year'})

   WUI 기간: 1952 ~ 2025
   WTUI 기간: 1996 ~ 2025


In [105]:
df_uncertainty.to_csv('data/output/uncertainty_annual.csv', index=False)